# Pipeline: damage_detection (modelo 1 - damaged / undamaged)

Lee los XML de `raw/`, decide si cada imagen esta danada o no, y copia las imagenes a `processed/damage_detection/{damaged,undamaged}/` (sin split train/val/test).

In [ ]:
import sys
sys.path.append('..')

from extract.extract_annotations import list_xml_keys, download_xml
from transform.parse_annotations import parse_binary_annotation
from load.load_to_s3 import copy_image

BUCKET = "humanoid-damage-cnn-834257582282-us-east-1-an"
RAW_ANNOTATIONS_PREFIX = "raw/IDEA/damages/annotations/"
RAW_IMAGES_PREFIX = "raw/IDEA/damages/images/"
PROCESSED_PREFIX = "processed/damage_detection/"
DRY_RUN = True  # cambiar a False cuando quieras subir de verdad

## Extract

In [ ]:
xml_keys = list_xml_keys(BUCKET, RAW_ANNOTATIONS_PREFIX)
print(f"Encontrados {len(xml_keys)} archivos XML.")

## Transform

In [ ]:
parsed = []
for key in xml_keys:
    try:
        xml_bytes = download_xml(BUCKET, key)
        annotation = parse_binary_annotation(xml_bytes)
    except Exception as exc:
        print(f"[ERROR] {key}: {exc}")
        continue
    if not annotation["image_basename"]:
        print(f"[ERROR] {key}: no se pudo determinar el nombre de la imagen")
        continue
    parsed.append(annotation)

n_damaged = sum(1 for a in parsed if a["is_damaged"])
n_undamaged = len(parsed) - n_damaged
print(f"Anotaciones parseadas: {len(parsed)}")
print(f"Damaged: {n_damaged} | Undamaged: {n_undamaged}")

## Load

In [ ]:
for i, annotation in enumerate(parsed, start=1):
    basename = annotation["image_basename"]
    source_key = f"{RAW_IMAGES_PREFIX}{basename}"
    clase_binaria = "damaged" if annotation["is_damaged"] else "undamaged"
    dest_key = f"{PROCESSED_PREFIX}{clase_binaria}/{basename}"
    copy_image(BUCKET, source_key, dest_key, dry_run=DRY_RUN)
    if i % 50 == 0 or i == len(parsed):
        print(f"Procesadas {i}/{len(parsed)} imagenes...")

print("Listo." if not DRY_RUN else "[DRY RUN] no se subio nada, poner DRY_RUN = False para ejecutar de verdad.")